In [1]:
import os
from dotenv import load_dotenv, find_dotenv

# Search upwards from current directory to find the root .env file
load_dotenv(find_dotenv())

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY not set — refusing to silently fall back to another provider")
if not MISTRAL_API_KEY:
    raise RuntimeError("MISTRAL_API_KEY not set — embeddings client cannot be built")

print("Environment OK")


Environment OK


## 1. Tools

In [2]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_mistralai import MistralAIEmbeddings

web_search_tool = DuckDuckGoSearchRun()
embeddings = MistralAIEmbeddings(model="mistral-embed")

print("Tools ready")

C:\Users\kmpra\AppData\Local\Temp\ipykernel_1820\2191445129.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun
e:\AI-Engineering\Practicals\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tools ready


In [3]:
# Standalone test — verify the search tool independently before wiring it in
print(web_search_tool.invoke("current RBI repo rate")[:500])

August 6, 2026 - What is Repo Rate? Meaning, Current Rates and Impact ... As per the announcement made by the Reserve Bank of India (RBI) on 05 August 2026, the current Repo Rate is 5.25%. August 7, 2026 - The Repo Rate is the interest rate at which the RBI lends money to commercial banks in exchange for securities to manage liquidity and control inflation. Currently at 5.25% (following a 25 basis points cut on 5 December 2025), the repo rate ... August 5, 2026 - India Holds Rates, Revises Down 


### Build or load the FAISS index

In [4]:
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

INDEX_DIR = "hr_policy_index"
# Resolve path to sample doc in exercise/docs/ or sample_docs/
SAMPLE_DOC_PATH = Path("docs/hr_policy_sample.txt")
if not SAMPLE_DOC_PATH.exists():
    SAMPLE_DOC_PATH = Path("../sample_docs/hr_policy_sample.txt")

def build_or_load_index() -> FAISS:
    if Path(INDEX_DIR).exists():
        return FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)

    if not SAMPLE_DOC_PATH.exists():
        raise RuntimeError(f"No index at {INDEX_DIR} and no source doc at {SAMPLE_DOC_PATH}")

    text = SAMPLE_DOC_PATH.read_text(encoding="utf-8")
    splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
    chunks = splitter.split_text(text)
    docs = [Document(page_content=chunk) for chunk in chunks]

    store = FAISS.from_documents(docs, embeddings)
    store.save_local(INDEX_DIR)
    return store

vectorstore = build_or_load_index()
print(f"Index ready with vectors: {vectorstore.index.ntotal}")


Index ready with vectors: 8


In [5]:
# Standalone test
for doc in vectorstore.similarity_search("notice period", k=2):
    print(doc.page_content[:200], "\n---")

1. Notice Period
All full-time employees must serve a notice period of 60 days upon
resignation, regardless of role or tenure. Employees with less than six
months of service may serve a reduced notice 
---
3. Leave and Notice Period Interaction
Employees serving a notice period may use accrued and carried-forward leave
during that period, subject to manager approval and adequate handover
coverage. Leave 
---


## 2. State schema

In [6]:
from typing import Annotated, Literal
from operator import add
from pydantic import BaseModel, Field, field_validator

class SubQuery(BaseModel):
    query: str
    source: str = Field(description="Must be 'rag' or 'web'")

    @field_validator("source")
    @classmethod
    def clean_source(cls, v: str) -> str:
        v_clean = v.strip().lower()
        if any(w in v_clean for w in ("rag", "internal", "company", "handbook", "policy")):
            return "rag"
        return "web"

class ResearchState(BaseModel):
    query: str
    plan: list[SubQuery] = Field(default_factory=list)
    round_docs: list[str] = Field(default_factory=list)
    sub_queries_done: Annotated[list[str], add] = Field(default_factory=list)
    retrieved_docs: Annotated[list[str], add] = Field(default_factory=list)
    summaries: Annotated[list[str], add] = Field(default_factory=list)
    iteration: int = 0
    sufficient: bool = False
    final_answer: str = ""

print("State schema ready")


State schema ready


## 3. Planner node

In [ ]:
from typing import Union
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

class PlanDecision(BaseModel):
    sub_queries: list[SubQuery] = Field(
        description="1-3 new, non-redundant sub-queries; empty if sufficient. "
                     "source='rag' for internal HR policy questions, "
                     "source='web' for anything current/external (rates, news, live data)."
    )
    # Using Union[bool, str] prevents Groq's tool validator from failing when Qwen outputs string 'False'
    sufficient: Union[bool, str] = Field(
        description="true or false. Set to true if summaries already answer the query."
    )
    reasoning: str

    @property
    def is_sufficient(self) -> bool:
        if isinstance(self.sufficient, str):
            return self.sufficient.strip().lower() in ("true", "1", "yes")
        return bool(self.sufficient)

plan_prompt = ChatPromptTemplate.from_template(
    """You are planning research to answer: {query}

Sub-queries already covered: {done}
Summaries collected so far:
{summaries}

For each new sub-query, tag its source as "rag" (internal HR policy docs)
or "web" (needs current/external information). Decide if summaries already
answer the query — if so, return sufficient=true and an empty sub_queries list."""
)

def plan_node(state: ResearchState) -> dict:
    summaries_text = "\n\n".join(state.summaries) if state.summaries else "None"
    structured_llm = llm.with_structured_output(PlanDecision)
    decision: PlanDecision = structured_llm.invoke(
        plan_prompt.format(
            query=state.query,
            done=state.sub_queries_done if state.sub_queries_done else "None",
            summaries=summaries_text,
        )
    )
    return {"plan": decision.sub_queries, "sufficient": decision.is_sufficient}


In [8]:
# Standalone test
test_state = ResearchState(query="What is the notice period for resignation?")
test_plan = plan_node(test_state)
print(test_plan)


{'plan': [SubQuery(query='notice period for resignation employee policy', source='rag'), SubQuery(query='resignation notice period requirements by role or contract type', source='rag')], 'sufficient': False}


## 4. Retrieval node (dispatcher)

In [9]:
def retrieve_node(state: ResearchState) -> dict:
    round_docs = []
    done = []

    for sq in state.plan:
        if sq.source == "rag":
            hits = vectorstore.similarity_search(sq.query, k=4)
            round_docs.extend(f"[RAG:{sq.query}] {doc.page_content}" for doc in hits)
        elif sq.source == "web":
            result = web_search_tool.invoke(sq.query)
            round_docs.append(f"[WEB:{sq.query}] {result}")
        else:
            raise ValueError(f"Unknown source tag: {sq.source}")
        done.append(sq.query)

    return {
        "round_docs": round_docs,
        "retrieved_docs": round_docs,
        "sub_queries_done": done,
    }

In [10]:
# Standalone test — reuse the plan from the previous test cell
test_state_2 = test_state.model_copy(update=test_plan)
print(retrieve_node(test_state_2))


{'round_docs': ['[RAG:notice period for resignation employee policy] 1. Notice Period\nAll full-time employees must serve a notice period of 60 days upon\nresignation, regardless of role or tenure. Employees with less than six\nmonths of service may serve a reduced notice period of 30 days, subject to\nmanager approval. During the notice period, employees remain eligible for\nall standard benefits and are expected to complete a formal knowledge', '[RAG:notice period for resignation employee policy] 3. Leave and Notice Period Interaction\nEmployees serving a notice period may use accrued and carried-forward leave\nduring that period, subject to manager approval and adequate handover\ncoverage. Leave taken during the notice period does not extend the notice\nperiod itself. Employees are encouraged to plan leave usage before\nsubmitting a resignation to avoid forfeiting accrued days.', '[RAG:notice period for resignation employee policy] 4. Remote Work\nEmployees may work remotely up to t

## 5. Summarizer node

In [11]:
summarize_prompt = ChatPromptTemplate.from_template(
    """Original question: {query}

Condense the following retrieved passages into a concise, factual summary.
Only include information relevant to the original question. No filler.

Passages:
{docs}"""
)

def summarize_node(state: ResearchState) -> dict:
    summary = llm.invoke(
        summarize_prompt.format(query=state.query, docs="\n\n".join(state.round_docs))
    ).content
    return {"summaries": [summary], "iteration": state.iteration + 1}

In [12]:
# Standalone test
test_state_3 = test_state_2.model_copy(update=retrieve_node(test_state_2))
print(summarize_node(test_state_3))

{'summaries': ['The standard notice period for resignation is 60 days for all full-time employees. However, the notice period is reduced to 30 days for employees with less than six months of service (subject to manager approval) and to 15 days for employees during their 90-day probation period.'], 'iteration': 1}


## Final answer node

In [13]:
answer_prompt = ChatPromptTemplate.from_template(
    """Answer the user's question using only the information in these summaries.
If they don't fully cover it, say so explicitly — don't fill gaps.

Question: {query}

Summaries:
{summaries}"""
)

def answer_node(state: ResearchState) -> dict:
    summaries_text = "\n\n".join(state.summaries) if state.summaries else "No information was retrieved."
    response = llm.invoke(
        answer_prompt.format(query=state.query, summaries=summaries_text)
    ).content
    return {"final_answer": response}


## 7. Routing logic

In [14]:
MAX_ITERATIONS = 3

def route_from_plan(state: ResearchState) -> str:
    # If the planner judged information sufficient or generated no sub-queries, answer now
    if state.sufficient or not state.plan:
        return "answer"
    return "retrieve"

def route_after_summarize(state: ResearchState) -> str:
    # If maximum iterations reached, go directly to answer without wasting an LLM planning call
    if state.iteration >= MAX_ITERATIONS:
        return "answer"
    return "planner" 


## 8. Build the graph

In [15]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ResearchState)
builder.add_node("planner", plan_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("summarize", summarize_node)
builder.add_node("answer", answer_node)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", route_from_plan, {"retrieve": "retrieve", "answer": "answer"})
builder.add_edge("retrieve", "summarize")
# Conditionally route after summarize to avoid wasted planning on max iterations
builder.add_conditional_edges("summarize", route_after_summarize, {"planner": "planner", "answer": "answer"})
builder.add_edge("answer", END)

print("Graph built")


Graph built


## 9. Persistence & run

In [18]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("research_checkpoints.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled with SQLite checkpointing")

Graph compiled with SQLite checkpointing


In [21]:
config = {"configurable": {"thread_id": "poc-session-001"}}

# Query is deliberately mixed: "our policy" forces source=rag,
# "typical industry practice" forces source=web — a smoke test that the
# planner's tagging actually works, not just that the graph compiles.
try:
    result = graph.invoke(
        {"query": "How does our current leave carry-forward policy compare to typical industry practice?"},
        config=config,
    )
    print(result["final_answer"])
except Exception as e:
    print(f"Run failed: {e}")
    # Resume: same thread_id, no new input — continues from the last
    # committed checkpoint rather than re-running the whole graph.
    result = graph.invoke(None, config=config)
    print(result["final_answer"])

Based on the provided summaries, here is how the current leave carry-forward policy compares to typical industry practice:

**1. Carry-Forward Cap**
*   **Current Policy:** Allows up to **10 unused days** to roll over.
*   **Industry Practice:** The typical limited carry-forward structure is usually **up to 5 days**. Therefore, the current policy is more generous than the typical industry standard for limited carry-forward models.

**2. Expiration/Use-By Date**
*   **Current Policy:** Carried-forward leave must be used within the **following calendar year** (i.e., by December 31st of the next year).
*   **Industry Practice:** Common structures often include a specific use-by date, such as **March 31** of the following year. The current policy allows for a longer usage window (the entire following year) compared to the common March 31 deadline.

**3. Forfeiture Rules ("Use-It-or-Lose-It")**
*   **Current Policy:** Implements a "use-it-or-lose-it" rule for days exceeding the 10-day cap, 